sensim+sv

In [10]:
import os
import re
import time
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from getpass import getpass

# =======================
# 🔧 CONFIG (EDIT HERE)
# =======================
TRAIN_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_LLMs.xlsx"
TEST_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_ChronicleA.xlsx"
OUTPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/ChronicleA_RESULTS"

MODEL_NAME = "gpt-4.1"
SIM_MODEL_NAME = "princeton-nlp/sup-simcse-bert-base-uncased"

k = 32
sentence_col = "sentence"
name_col = "gold_names"

SLEEP_SECONDS = 2


# =======================
# 🔐 API KEY
# =======================
api_key = getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)


# =======================
# 🧠 LOAD MODEL
# =======================
sim_model = SentenceTransformer(SIM_MODEL_NAME, device="cpu")


# =======================
# 🧩 HELPER FUNCTIONS
# =======================
def normalize_text(s):
    if pd.isna(s):
        return ""
    return " ".join(str(s).strip().split())


def normalize_entity(s):
    return s.casefold().strip()


def parse_gold_names(cell):
    if pd.isna(cell):
        return []
    text = str(cell)
    parts = re.split(r"[;,|]", text)
    return [p.strip() for p in parts if p.strip()]


def extract_marked_names(text):
    return list(set(re.findall(r"@@(.*?)##", text)))


def mark_gold_names(sentence, names):
    for name in sorted(names, key=len, reverse=True):
        pattern = re.escape(name)
        sentence = re.sub(pattern, f"@@{name}##", sentence)
    return sentence


def encode_sentences(sentences):
    return sim_model.encode(sentences, normalize_embeddings=True)


def get_topk(train_sents, train_embs, test_sent):
    test_emb = sim_model.encode([test_sent], normalize_embeddings=True)
    sims = cosine_similarity(test_emb, train_embs)[0]
    topk_idx = np.argsort(sims)[-k:][::-1]
    return topk_idx


def build_prompt(test_sentence, demo_examples):
    prompt = "I am an excellent linguist.\n\n"
    prompt += "Label person entities using @@ and ##.\n\n"

    for inp, out in demo_examples:
        prompt += f"Input: {inp}\nOutput: {out}\n\n"

    prompt += f"Input: {test_sentence}\nOutput:"
    return prompt


def call_gpt(prompt, model):
    response = client.responses.create(
        model=model,
        input=prompt
    )
    return response.output[0].content[0].text


def verify_entities(sentence, pred_names):
    verified = []

    for name in pred_names:
        prompt = f"""
Sentence: {sentence}
Entity: {name}

Is this a correct person entity? Answer Yes or No.
"""
        ans = call_gpt(prompt, MODEL_NAME).lower()
        if "yes" in ans:
            verified.append(name)

    return verified


def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return precision, recall, f1


# =======================
# 📂 LOAD DATA
# =======================
train_df = pd.read_excel(TRAIN_FILE)
test_df = pd.read_excel(TEST_FILE)

train_df[sentence_col] = train_df[sentence_col].apply(normalize_text)
test_df[sentence_col] = test_df[sentence_col].apply(normalize_text)

train_sents = train_df[sentence_col].tolist()
train_embs = encode_sentences(train_sents)


# =======================
# 🚀 MAIN LOOP
# =======================
results = []
tp = fp = fn = 0

for i, row in test_df.iterrows():
    sentence = row[sentence_col]
    gold_names = parse_gold_names(row[name_col])

    # 🔍 retrieval
    topk_idx = get_topk(train_sents, train_embs, sentence)

    demo_examples = []
    for idx in topk_idx:
        sent = train_sents[idx]
        names = parse_gold_names(train_df.iloc[idx][name_col])
        marked = mark_gold_names(sent, names)
        demo_examples.append((sent, marked))

    # 🤖 first prediction
    prompt = build_prompt(sentence, demo_examples)
    pred_output = call_gpt(prompt, MODEL_NAME)

    pred_names = extract_marked_names(pred_output)

    # 🔁 self verification
    verified_names = verify_entities(sentence, pred_names)

    # final output rebuild
    final_output = sentence
    for name in verified_names:
        final_output = re.sub(re.escape(name), f"@@{name}##", final_output)

    # 📊 metrics
    gold_set = set(map(normalize_entity, gold_names))
    pred_set = set(map(normalize_entity, verified_names))

    row_tp = len(gold_set & pred_set)
    row_fp = len(pred_set - gold_set)
    row_fn = len(gold_set - pred_set)

    tp += row_tp
    fp += row_fp
    fn += row_fn

    results.append({
        "sentence": sentence,
        "gold": "; ".join(gold_names),
        "pred": "; ".join(pred_names),
        "verified": "; ".join(verified_names),
        "tp": row_tp,
        "fp": row_fp,
        "fn": row_fn
    })

    print(f"[{i}] TP={row_tp} FP={row_fp} FN={row_fn}")
    time.sleep(SLEEP_SECONDS)


# =======================
# 📊 FINAL METRICS
# =======================
precision, recall, f1 = compute_metrics(tp, fp, fn)

print("\n=== FINAL ===")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)


# =======================
# 💾 SAVE OUTPUT
# =======================
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame(results).to_excel(
    os.path.join(OUTPUT_DIR, "SSim_SV_k32_predictions.xlsx"),
    index=False
)

with open(os.path.join(OUTPUT_DIR, "SSim_SV_k32_final_scores.txt"), "w") as f:
    f.write(f"Precision: {precision}\n")
    f.write(f"Recall: {recall}\n")
    f.write(f"F1: {f1}\n")

print("\nSaved results to:", OUTPUT_DIR)

No sentence-transformers model found with name princeton-nlp/sup-simcse-bert-base-uncased. Creating a new one with mean pooling.


[0] TP=2 FP=0 FN=0
[1] TP=1 FP=0 FN=0
[2] TP=3 FP=0 FN=0
[3] TP=2 FP=0 FN=1
[4] TP=2 FP=0 FN=1
[5] TP=1 FP=1 FN=0
[6] TP=1 FP=0 FN=0
[7] TP=1 FP=1 FN=0
[8] TP=1 FP=0 FN=0
[9] TP=1 FP=1 FN=0
[10] TP=1 FP=1 FN=0
[11] TP=1 FP=0 FN=0
[12] TP=1 FP=1 FN=0
[13] TP=2 FP=0 FN=0
[14] TP=1 FP=0 FN=0
[15] TP=1 FP=0 FN=0
[16] TP=1 FP=0 FN=0
[17] TP=1 FP=0 FN=0
[18] TP=1 FP=0 FN=1
[19] TP=2 FP=0 FN=0
[20] TP=2 FP=0 FN=1
[21] TP=1 FP=0 FN=1
[22] TP=0 FP=1 FN=2
[23] TP=2 FP=0 FN=1
[24] TP=0 FP=0 FN=1
[25] TP=1 FP=0 FN=0
[26] TP=6 FP=0 FN=0
[27] TP=1 FP=0 FN=0
[28] TP=2 FP=0 FN=0
[29] TP=1 FP=0 FN=0
[30] TP=1 FP=0 FN=0
[31] TP=1 FP=0 FN=0
[32] TP=2 FP=0 FN=0
[33] TP=1 FP=0 FN=1
[34] TP=1 FP=0 FN=0
[35] TP=1 FP=0 FN=0
[36] TP=1 FP=0 FN=0
[37] TP=1 FP=1 FN=0
[38] TP=0 FP=0 FN=1
[39] TP=0 FP=2 FN=1
[40] TP=1 FP=0 FN=0
[41] TP=0 FP=0 FN=2
[42] TP=1 FP=0 FN=0
[43] TP=2 FP=0 FN=0
[44] TP=1 FP=0 FN=0
[45] TP=1 FP=0 FN=0
[46] TP=2 FP=0 FN=0
[47] TP=1 FP=0 FN=0
[48] TP=1 FP=0 FN=0
[49] TP=2 FP=0 FN=1
[50] TP=1 

sm without sv

In [11]:
import os
import re
import time
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from getpass import getpass

# =======================
# CONFIG (EDIT HERE)
# =======================
TRAIN_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_LLMs.xlsx"
TEST_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_ChronicleA.xlsx"
OUTPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/ChronicleA_RESULTS/#SV"

MODEL_NAME = "gpt-4.1"
SIM_MODEL_NAME = "princeton-nlp/sup-simcse-bert-base-uncased"

k = 32
sentence_col = "sentence"
name_col = "gold_names"

SLEEP_SECONDS = 2


# =======================
# API KEY
# =======================
api_key = getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)


# =======================
# LOAD MODEL
# =======================
sim_model = SentenceTransformer(SIM_MODEL_NAME, device="cpu")


# =======================
# HELPER FUNCTIONS
# =======================
def normalize_text(s):
    if pd.isna(s):
        return ""
    return " ".join(str(s).strip().split())


def normalize_entity(s):
    return s.casefold().strip()


def parse_gold_names(cell):
    if pd.isna(cell):
        return []
    text = str(cell)
    parts = re.split(r"[;,|]", text)
    return [p.strip() for p in parts if p.strip()]


def extract_marked_names(text):
    matches = re.findall(r"@@(.*?)##", str(text))
    seen = set()
    result = []
    for m in matches:
        m = m.strip()
        if m and m not in seen:
            seen.add(m)
            result.append(m)
    return result


def mark_gold_names(sentence, names):
    sentence = str(sentence)
    for name in sorted(names, key=len, reverse=True):
        pattern = re.escape(name)
        sentence = re.sub(pattern, f"@@{name}##", sentence)
    return sentence


def encode_sentences(sentences):
    return sim_model.encode(
        sentences,
        normalize_embeddings=True
    )


def get_topk(train_sents, train_embs, test_sent):
    test_emb = sim_model.encode([test_sent], normalize_embeddings=True)
    sims = cosine_similarity(test_emb, train_embs)[0]
    topk_idx = np.argsort(sims)[-k:][::-1]
    topk_scores = sims[topk_idx]
    return topk_idx, topk_scores


def build_prompt(test_sentence, demo_examples):
    prompt_lines = [
        "I am an excellent linguist.",
        "",
        "The task is to label person entities in the given Old English sentence using @@ and ##.",
        "If no person entity is present, return the sentence unchanged.",
        "",
        "Below are some examples.",
        ""
    ]

    for inp, out in demo_examples:
        prompt_lines.append(f"Input: {inp}")
        prompt_lines.append(f"Output: {out}")
        prompt_lines.append("")

    prompt_lines.append(f"Input: {test_sentence}")
    prompt_lines.append("Output:")

    return "\n".join(prompt_lines)


def call_gpt(prompt, model):
    response = client.responses.create(
        model=model,
        input=prompt
    )
    return response.output[0].content[0].text


def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1


# =======================
# LOAD DATA
# =======================
train_df = pd.read_excel(TRAIN_FILE)
test_df = pd.read_excel(TEST_FILE)

train_df[sentence_col] = train_df[sentence_col].apply(normalize_text)
test_df[sentence_col] = test_df[sentence_col].apply(normalize_text)

train_sents = train_df[sentence_col].tolist()
train_embs = encode_sentences(train_sents)


# =======================
# MAIN LOOP
# =======================
results = []
tp = fp = fn = 0

for i, row in test_df.iterrows():
    sentence = row[sentence_col]
    gold_names = parse_gold_names(row[name_col])

    # retrieval
    topk_idx, topk_scores = get_topk(train_sents, train_embs, sentence)

    demo_examples = []
    retrieved_sentences = []
    retrieved_scores = []

    for idx, score in zip(topk_idx, topk_scores):
        sent = train_sents[idx]
        names = parse_gold_names(train_df.iloc[idx][name_col])
        marked = mark_gold_names(sent, names)

        demo_examples.append((sent, marked))
        retrieved_sentences.append(sent)
        retrieved_scores.append(float(score))

    # prediction only (no self-verification)
    prompt = build_prompt(sentence, demo_examples)
    pred_output = call_gpt(prompt, MODEL_NAME)
    pred_names = extract_marked_names(pred_output)

    # metrics
    gold_set = set(map(normalize_entity, gold_names))
    pred_set = set(map(normalize_entity, pred_names))

    row_tp = len(gold_set & pred_set)
    row_fp = len(pred_set - gold_set)
    row_fn = len(gold_set - pred_set)

    tp += row_tp
    fp += row_fp
    fn += row_fn

    row_result = {
        "sentence": sentence,
        "gold_name_raw": row[name_col],
        "gold_names_parsed": "; ".join(gold_names),
        "model_output": pred_output,
        "pred_names_parsed": "; ".join(pred_names),
        "row_tp": row_tp,
        "row_fp": row_fp,
        "row_fn": row_fn,
    }

    for j in range(k):
        row_result[f"retrieved_demo_{j+1}"] = retrieved_sentences[j] if j < len(retrieved_sentences) else ""
        row_result[f"retrieval_score_{j+1}"] = retrieved_scores[j] if j < len(retrieved_scores) else ""

    results.append(row_result)

    print(f"[{i}] TP={row_tp} FP={row_fp} FN={row_fn}")
    time.sleep(SLEEP_SECONDS)


# =======================
# FINAL METRICS
# =======================
precision, recall, f1 = compute_metrics(tp, fp, fn)

print("\n=== FINAL ===")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)


# =======================
# SAVE OUTPUT
# =======================
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame(results).to_excel(
    os.path.join(OUTPUT_DIR, "SSim_k32_predictions.xlsx"),
    index=False
)

summary_df = pd.DataFrame([{
    "tp": tp,
    "fp": fp,
    "fn": fn,
    "precision": precision,
    "recall": recall,
    "f1": f1
}])

summary_df.to_excel(
    os.path.join(OUTPUT_DIR, "SSim_k32_metrics_summary.xlsx"),
    index=False
)

with open(os.path.join(OUTPUT_DIR, "SSim_k32_final_scores.txt"), "w") as f:
    f.write(f"TP: {tp}\n")
    f.write(f"FP: {fp}\n")
    f.write(f"FN: {fn}\n")
    f.write(f"Precision: {precision}\n")
    f.write(f"Recall: {recall}\n")
    f.write(f"F1: {f1}\n")

print("\nSaved results to:", OUTPUT_DIR)

No sentence-transformers model found with name princeton-nlp/sup-simcse-bert-base-uncased. Creating a new one with mean pooling.


[0] TP=2 FP=0 FN=0
[1] TP=1 FP=0 FN=0
[2] TP=3 FP=0 FN=0
[3] TP=3 FP=0 FN=0
[4] TP=3 FP=0 FN=0
[5] TP=1 FP=1 FN=0
[6] TP=1 FP=1 FN=0
[7] TP=1 FP=1 FN=0
[8] TP=0 FP=1 FN=1
[9] TP=1 FP=1 FN=0
[10] TP=1 FP=1 FN=0
[11] TP=1 FP=1 FN=0
[12] TP=1 FP=0 FN=0
[13] TP=2 FP=0 FN=0
[14] TP=1 FP=0 FN=0
[15] TP=1 FP=0 FN=0
[16] TP=1 FP=0 FN=0
[17] TP=1 FP=0 FN=0
[18] TP=2 FP=0 FN=0
[19] TP=2 FP=0 FN=0
[20] TP=3 FP=0 FN=0
[21] TP=2 FP=0 FN=0
[22] TP=2 FP=0 FN=0
[23] TP=3 FP=1 FN=0
[24] TP=1 FP=0 FN=0
[25] TP=1 FP=0 FN=0
[26] TP=6 FP=0 FN=0
[27] TP=1 FP=0 FN=0
[28] TP=2 FP=0 FN=0
[29] TP=1 FP=0 FN=0
[30] TP=1 FP=0 FN=0
[31] TP=1 FP=0 FN=0
[32] TP=2 FP=0 FN=0
[33] TP=2 FP=0 FN=0
[34] TP=1 FP=0 FN=0
[35] TP=1 FP=0 FN=0
[36] TP=1 FP=0 FN=0
[37] TP=1 FP=1 FN=0
[38] TP=1 FP=0 FN=0
[39] TP=0 FP=4 FN=1
[40] TP=1 FP=0 FN=0
[41] TP=2 FP=0 FN=0
[42] TP=1 FP=0 FN=0
[43] TP=1 FP=1 FN=1
[44] TP=1 FP=0 FN=0
[45] TP=1 FP=0 FN=0
[46] TP=1 FP=2 FN=1
[47] TP=1 FP=0 FN=0
[48] TP=1 FP=1 FN=0
[49] TP=3 FP=0 FN=0
[50] TP=1 

sm+sv - correct prompt

In [12]:
import os
import re
import time
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from getpass import getpass

# =======================
# CONFIG
# =======================
TRAIN_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_LLMs.xlsx"
TEST_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_ChronicleA.xlsx"
OUTPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/ChronicleA_Results/SenSim+SV"

MODEL_NAME = "gpt-4.1"
VERIFY_MODEL_NAME = "gpt-4.1"
SIM_MODEL_NAME = "princeton-nlp/sup-simcse-bert-base-uncased"

k = 32
sentence_col = "sentence"
name_col = "gold_names"

SLEEP_SECONDS = 2


# =======================
# API KEY
# =======================
api_key = getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)


# =======================
# LOAD MODEL
# =======================
sim_model = SentenceTransformer(SIM_MODEL_NAME, device="cpu")


# =======================
# HELPERS
# =======================
def normalize_text(s):
    if pd.isna(s):
        return ""
    return " ".join(str(s).strip().split())


def normalize_entity(s):
    return s.casefold().strip()


def parse_gold_names(cell):
    if pd.isna(cell):
        return []
    parts = re.split(r"[;,|]", str(cell))
    return [p.strip() for p in parts if p.strip()]


def extract_marked_names(text):
    matches = re.findall(r"@@(.*?)##", str(text))
    seen, result = set(), []
    for m in matches:
        m = m.strip()
        if m and m not in seen:
            seen.add(m)
            result.append(m)
    return result


def mark_gold_names(sentence, names):
    for name in sorted(names, key=len, reverse=True):
        sentence = re.sub(re.escape(name), f"@@{name}##", sentence)
    return sentence


def encode_sentences(sentences):
    return sim_model.encode(sentences, normalize_embeddings=True)


def get_topk(train_sents, train_embs, test_sent):
    test_emb = sim_model.encode([test_sent], normalize_embeddings=True)
    sims = cosine_similarity(test_emb, train_embs)[0]
    topk_idx = np.argsort(sims)[-k:][::-1]
    return topk_idx


# =======================
# 🔥 ORIGINAL PROMPT
# =======================
def build_prompt(test_sentence, demo_examples):
    lines = [
        "I am an excellent linguist.",
        "",
        "The task is to label person entities in the given Old English sentence using @@ and ##.",
        "If no person entity is present, return the sentence unchanged.",
        "",
        "Below are some examples.",
        ""
    ]

    for inp, out in demo_examples:
        lines.append(f"Input: {inp}")
        lines.append(f"Output: {out}")
        lines.append("")

    lines.append(f"Input: {test_sentence}")
    lines.append("Output:")

    return "\n".join(lines)


# =======================
# 🔥 ORIGINAL SV PROMPT
# =======================
def verify_entities(sentence, pred_names):
    verified = []

    for word in pred_names:
        prompt = f"""
Sentence: {sentence}

Word: {word}

Is "{word}" referring to a person name in this sentence?
Answer only Yes or No.
"""
        response = client.responses.create(
            model=VERIFY_MODEL_NAME,
            input=prompt
        )
        ans = response.output[0].content[0].text.lower()

        if "yes" in ans:
            verified.append(word)

    return verified


def call_gpt(prompt):
    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt
    )
    return response.output[0].content[0].text


def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return precision, recall, f1


# =======================
# LOAD DATA
# =======================
train_df = pd.read_excel(TRAIN_FILE)
test_df = pd.read_excel(TEST_FILE)

train_df[sentence_col] = train_df[sentence_col].apply(normalize_text)
test_df[sentence_col] = test_df[sentence_col].apply(normalize_text)

train_sents = train_df[sentence_col].tolist()
train_embs = encode_sentences(train_sents)


# =======================
# MAIN LOOP
# =======================
results = []
tp = fp = fn = 0

for i, row in test_df.iterrows():
    sentence = row[sentence_col]
    gold_names = parse_gold_names(row[name_col])

    # retrieval
    topk_idx = get_topk(train_sents, train_embs, sentence)

    demo_examples = []
    for idx in topk_idx:
        sent = train_sents[idx]
        names = parse_gold_names(train_df.iloc[idx][name_col])
        marked = mark_gold_names(sent, names)
        demo_examples.append((sent, marked))

    # prediction
    prompt = build_prompt(sentence, demo_examples)
    pred_output = call_gpt(prompt)
    pred_names = extract_marked_names(pred_output)

    # self verification
    verified_names = verify_entities(sentence, pred_names)

    # metrics
    gold_set = set(map(normalize_entity, gold_names))
    pred_set = set(map(normalize_entity, verified_names))

    row_tp = len(gold_set & pred_set)
    row_fp = len(pred_set - gold_set)
    row_fn = len(gold_set - pred_set)

    tp += row_tp
    fp += row_fp
    fn += row_fn

    results.append({
        "sentence": sentence,
        "gold": "; ".join(gold_names),
        "pred": "; ".join(pred_names),
        "verified": "; ".join(verified_names),
        "tp": row_tp,
        "fp": row_fp,
        "fn": row_fn
    })

    print(f"[{i}] TP={row_tp} FP={row_fp} FN={row_fn}")
    time.sleep(SLEEP_SECONDS)


# =======================
# FINAL METRICS
# =======================
precision, recall, f1 = compute_metrics(tp, fp, fn)

print("\n=== FINAL ===")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)


# =======================
# SAVE
# =======================
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame(results).to_excel(
    os.path.join(OUTPUT_DIR, "SSim_SV_k32_predictions.xlsx"),
    index=False
)

with open(os.path.join(OUTPUT_DIR, "SSim_SV_k32_final_scores.txt"), "w") as f:
    f.write(f"Precision: {precision}\nRecall: {recall}\nF1: {f1}\n")

print("\nSaved to:", OUTPUT_DIR)

No sentence-transformers model found with name princeton-nlp/sup-simcse-bert-base-uncased. Creating a new one with mean pooling.


[0] TP=2 FP=0 FN=0
[1] TP=1 FP=0 FN=0
[2] TP=3 FP=0 FN=0
[3] TP=3 FP=0 FN=0
[4] TP=3 FP=0 FN=0
[5] TP=1 FP=1 FN=0
[6] TP=1 FP=1 FN=0
[7] TP=1 FP=0 FN=0
[8] TP=1 FP=1 FN=0
[9] TP=1 FP=1 FN=0
[10] TP=1 FP=1 FN=0
[11] TP=1 FP=1 FN=0
[12] TP=1 FP=1 FN=0
[13] TP=2 FP=0 FN=0
[14] TP=1 FP=0 FN=0
[15] TP=1 FP=0 FN=0
[16] TP=1 FP=0 FN=0
[17] TP=1 FP=0 FN=0
[18] TP=1 FP=0 FN=1
[19] TP=2 FP=0 FN=0
[20] TP=3 FP=0 FN=0
[21] TP=2 FP=0 FN=0
[22] TP=2 FP=0 FN=0
[23] TP=3 FP=0 FN=0
[24] TP=1 FP=0 FN=0
[25] TP=1 FP=0 FN=0
[26] TP=6 FP=0 FN=0
[27] TP=1 FP=0 FN=0
[28] TP=2 FP=0 FN=0
[29] TP=1 FP=0 FN=0
[30] TP=1 FP=0 FN=0
[31] TP=1 FP=0 FN=0
[32] TP=2 FP=0 FN=0
[33] TP=1 FP=0 FN=1
[34] TP=1 FP=0 FN=0
[35] TP=1 FP=0 FN=0
[36] TP=1 FP=0 FN=0
[37] TP=1 FP=1 FN=0
[38] TP=1 FP=0 FN=0
[39] TP=0 FP=2 FN=1
[40] TP=1 FP=0 FN=0
[41] TP=2 FP=0 FN=0
[42] TP=1 FP=0 FN=0
[43] TP=1 FP=0 FN=1
[44] TP=1 FP=0 FN=0
[45] TP=1 FP=0 FN=0
[46] TP=2 FP=0 FN=0
[47] TP=1 FP=0 FN=0
[48] TP=1 FP=0 FN=0
[49] TP=2 FP=0 FN=1
[50] TP=1 

sm+sv final


In [15]:
import os
import re
import time
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from getpass import getpass

# =======================
# CONFIG
# =======================
TRAIN_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_LLMs.xlsx"
TEST_FILE = "/Users/shinekhantaung/Desktop/GPT-NER/D_ChronicleA.xlsx"
OUTPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/ChronicleA_Results/SenSim+SV_final"

MODEL_NAME = "gpt-4.1"
VERIFY_MODEL_NAME = "gpt-4.1"
SIM_MODEL_NAME = "princeton-nlp/sup-simcse-bert-base-uncased"

k = 32
sentence_col = "sentence"
name_col = "gold_names"

SLEEP_SECONDS = 2


# =======================
# API KEY
# =======================
api_key = getpass("Enter OpenAI API key: ")
client = OpenAI(api_key=api_key)


# =======================
# LOAD MODEL
# =======================
sim_model = SentenceTransformer(SIM_MODEL_NAME, device="cpu")


# =======================
# HELPERS
# =======================
def normalize_text(s):
    if pd.isna(s):
        return ""
    return " ".join(str(s).strip().split())


def normalize_entity(s):
    return s.casefold().strip()


def parse_gold_names(cell):
    if pd.isna(cell):
        return []
    parts = re.split(r"[;,|]", str(cell))
    return [p.strip() for p in parts if p.strip()]


def extract_marked_names(text):
    matches = re.findall(r"@@(.*?)##", str(text))
    seen, result = set(), []
    for m in matches:
        m = m.strip()
        if m and m not in seen:
            seen.add(m)
            result.append(m)
    return result


def mark_gold_names(sentence, names):
    for name in sorted(names, key=len, reverse=True):
        sentence = re.sub(re.escape(name), f"@@{name}##", sentence)
    return sentence


def encode_sentences(sentences):
    return sim_model.encode(sentences, normalize_embeddings=True)


def get_topk(train_sents, train_embs, test_sent):
    test_emb = sim_model.encode([test_sent], normalize_embeddings=True)
    sims = cosine_similarity(test_emb, train_embs)[0]
    topk_idx = np.argsort(sims)[-k:][::-1]
    return topk_idx


# =======================
# 🔥 ORIGINAL PROMPT
# =======================
def build_prompt(test_sentence, demo_examples):
    lines = [
        "I am an excellent linguist.",
        "",
        "The task is to label person entities in the given Old English sentence using @@ and ##.",
        "If no person entity is present, return the sentence unchanged.",
        "",
        "Below are some examples.",
        ""
    ]

    for inp, out in demo_examples:
        lines.append(f"Input: {inp}")
        lines.append(f"Output: {out}")
        lines.append("")

    lines.append(f"Input: {test_sentence}")
    lines.append("Output:")

    return "\n".join(lines)


# =======================
# 🔥 ORIGINAL SV PROMPT
# =======================
def verify_entities(sentence, pred_names):
    verified = []

    for word in pred_names:
        prompt = f"""
Sentence: {sentence}

Word: {word}

Is "{word}" referring to a person name in this sentence?
Answer only Yes or No.
"""
        response = client.responses.create(
            model=VERIFY_MODEL_NAME,
            input=prompt
        )
        ans = response.output[0].content[0].text.lower()

        if "yes" in ans:
            verified.append(word)

    return verified


def call_gpt(prompt):
    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt
    )
    return response.output[0].content[0].text


def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return precision, recall, f1


# =======================
# LOAD DATA
# =======================
train_df = pd.read_excel(TRAIN_FILE)
test_df = pd.read_excel(TEST_FILE)

train_df[sentence_col] = train_df[sentence_col].apply(normalize_text)
test_df[sentence_col] = test_df[sentence_col].apply(normalize_text)

train_sents = train_df[sentence_col].tolist()
train_embs = encode_sentences(train_sents)


# =======================
# MAIN LOOP
# =======================
results = []
tp = fp = fn = 0

for i, row in test_df.iterrows():
    sentence = row[sentence_col]
    gold_names = parse_gold_names(row[name_col])

    # retrieval
    topk_idx = get_topk(train_sents, train_embs, sentence)

    demo_examples = []
    for idx in topk_idx:
        sent = train_sents[idx]
        names = parse_gold_names(train_df.iloc[idx][name_col])
        marked = mark_gold_names(sent, names)
        demo_examples.append((sent, marked))

    # prediction
    prompt = build_prompt(sentence, demo_examples)
    pred_output = call_gpt(prompt)
    pred_names = extract_marked_names(pred_output)

    # self verification
    verified_names = verify_entities(sentence, pred_names)

    # metrics
    gold_set = set(map(normalize_entity, gold_names))
    pred_set = set(map(normalize_entity, verified_names))

    row_tp = len(gold_set & pred_set)
    row_fp = len(pred_set - gold_set)
    row_fn = len(gold_set - pred_set)

    tp += row_tp
    fp += row_fp
    fn += row_fn

    results.append({
        "sentence": sentence,
        "gold": "; ".join(gold_names),
        "pred": "; ".join(pred_names),
        "verified": "; ".join(verified_names),
        "tp": row_tp,
        "fp": row_fp,
        "fn": row_fn
    })

    print(f"[{i}] TP={row_tp} FP={row_fp} FN={row_fn}")
    time.sleep(SLEEP_SECONDS)


# =======================
# FINAL METRICS
# =======================
precision, recall, f1 = compute_metrics(tp, fp, fn)

print("\n=== FINAL ===")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)


# =======================
# SAVE
# =======================
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame(results).to_excel(
    os.path.join(OUTPUT_DIR, "SSim_SV_k32_predictions.xlsx"),
    index=False
)

with open(os.path.join(OUTPUT_DIR, "SSim_SV_k32_final_scores.txt"), "w") as f:
    f.write(f"Precision: {precision}\nRecall: {recall}\nF1: {f1}\n")

print("\nSaved to:", OUTPUT_DIR)

No sentence-transformers model found with name princeton-nlp/sup-simcse-bert-base-uncased. Creating a new one with mean pooling.


[0] TP=2 FP=0 FN=0
[1] TP=1 FP=0 FN=0
[2] TP=3 FP=0 FN=0
[3] TP=3 FP=0 FN=0
[4] TP=3 FP=0 FN=0
[5] TP=2 FP=0 FN=0
[6] TP=1 FP=1 FN=0
[7] TP=1 FP=0 FN=1
[8] TP=2 FP=0 FN=0
[9] TP=2 FP=0 FN=0
[10] TP=2 FP=0 FN=0
[11] TP=2 FP=0 FN=0
[12] TP=2 FP=0 FN=0
[13] TP=2 FP=0 FN=0
[14] TP=1 FP=0 FN=0
[15] TP=1 FP=0 FN=0
[16] TP=1 FP=0 FN=0
[17] TP=1 FP=0 FN=0
[18] TP=1 FP=0 FN=1
[19] TP=2 FP=0 FN=0
[20] TP=2 FP=1 FN=0
[21] TP=2 FP=0 FN=0
[22] TP=0 FP=2 FN=1
[23] TP=3 FP=0 FN=0
[24] TP=1 FP=0 FN=0
[25] TP=1 FP=0 FN=0
[26] TP=6 FP=0 FN=0
[27] TP=1 FP=0 FN=0
[28] TP=2 FP=0 FN=0
[29] TP=1 FP=0 FN=0
[30] TP=1 FP=0 FN=0
[31] TP=1 FP=0 FN=0
[32] TP=2 FP=0 FN=0
[33] TP=1 FP=0 FN=1
[34] TP=1 FP=0 FN=0
[35] TP=1 FP=0 FN=0
[36] TP=1 FP=0 FN=0
[37] TP=2 FP=0 FN=0
[38] TP=1 FP=0 FN=0
[39] TP=2 FP=0 FN=0
[40] TP=1 FP=0 FN=0
[41] TP=2 FP=0 FN=0
[42] TP=1 FP=0 FN=0
[43] TP=2 FP=0 FN=0
[44] TP=1 FP=0 FN=0
[45] TP=1 FP=0 FN=0
[46] TP=2 FP=0 FN=0
[47] TP=1 FP=0 FN=0
[48] TP=1 FP=0 FN=0
[49] TP=2 FP=0 FN=1
[50] TP=1 